In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-04 08:21:21.512655: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-04 08:21:22.230267: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {}
      
    },
  "temp_data_path": "../../../../",
  "partitions": 1,
  "num_of_workers": 1,
  "iterations": 2,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-04 08:21:23,216 [DEBUG] [Rain] Rain is initialized
2023-07-04 08:21:23,217 [DEBUG] [Provisioner] Creating coordinator
2023-07-04 08:21:23,218 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 08:21:23,219 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-04 08:21:23,220 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-04 08:21:23,221 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 08:21:23,222 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-04 08:21:23,223 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 08:21:23,225 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 08:21:23,225 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-04 08:21:23,232 [DEBUG] [Rain] Creating workers
2023-07-04 08:21:23,239 [INFO] [Provisioner] provisioner is serving
2023-07-04 08:21:23,240 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 08:21:23,242 [INFO] [Coordinator] coordinator is serving
2023-07-04 08:21:23,243 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 08:21:23,247 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 08:21:23,248 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 08:21:23,249 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 08:21:23,250 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 08:21:23,253 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 08:21:23,254 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 08:21:23,256 [INFO] [W

469/469 [==============================] - 3s 4ms/step - loss: 0.4239 - accuracy: 0.8709
sending data to coordinator


2023-07-04 08:21:45,256 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-04 08:21:45,257 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-04 08:21:45,336 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 08:21:45,351 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
2023-07-04 08:21:45,378 [DEBUG] [DeepLearning] Iteration 1/2 complete for worker 1.
2023-07-04 08:21:45,379 [DEBUG] [DeepLearning] Starting iteration 2/2
2023-07-04 08:21:45,380 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-04 08:21:45,380 [DEBUG] [DividerAmbassador] Sending ../../../..//RainData/divider/1.pkl to worker1
2023-07-04 08:21:45,458 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-04 08:21:45,459 [DEBUG] [DividerAmbassador] divide

469/469 [==============================] - 2s 3ms/step - loss: 0.1986 - accuracy: 0.9408
sending data to coordinator


2023-07-04 08:21:50,825 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-04 08:21:50,826 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-04 08:21:50,897 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 08:21:50,903 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
2023-07-04 08:21:50,921 [DEBUG] [DeepLearning] Iteration 2/2 complete for worker 1.
2023-07-04 08:21:50,922 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-04 08:21:50,922 [DEBUG] [Divider] Divider stopped serving
2023-07-04 08:21:50,923 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-04 08:21:50,924 [DEBUG] [Divider] Divider stopped serving
2023-07-04 08:21:50,924 [INFO] [Provisioner] provisioner stopped serving


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.1066 - accuracy: 0.9668

Test accuracy: 96.7%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-04 08:21:51,413 [DEBUG] [Rain] Creating workers
2023-07-04 08:21:51,415 [INFO] [Provisioner] provisioner is serving
2023-07-04 08:21:51,415 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 08:21:51,417 [INFO] [Coordinator] coordinator is serving
2023-07-04 08:21:51,417 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 08:21:51,419 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 08:21:51,420 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 08:21:51,421 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 08:21:51,422 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 08:21:51,560 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 08:21:51,560 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 08:21:51,562 [DEBUG] [TemporaryFilesManager] Creating temp

469/469 [==============================] - 2s 3ms/step - loss: 0.1528 - accuracy: 0.9552
sending data to coordinator


2023-07-04 08:22:11,032 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-04 08:22:11,033 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-04 08:22:11,110 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 08:22:11,134 [DEBUG] [DeepLearning] Iteration 1/2 complete.
2023-07-04 08:22:11,135 [DEBUG] [DeepLearning] Starting iteration 2/2
2023-07-04 08:22:11,162 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-04 08:22:11,163 [DEBUG] [DividerAmbassador] Sending ../../../..//RainData/divider/1.pkl to worker1
2023-07-04 08:22:11,246 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-04 08:22:11,247 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker1
2023-07-04 08:22:11,247 [INFO] [Worker_50151] Executing co

469/469 [==============================] - 2s 2ms/step - loss: 0.1299 - accuracy: 0.9612
sending data to coordinator


2023-07-04 08:22:16,057 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-04 08:22:16,058 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_2_trained.pkl from worker1
2023-07-04 08:22:16,131 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_2_trained.pkl from worker1 successfully
2023-07-04 08:22:16,151 [DEBUG] [DeepLearning] Iteration 2/2 complete.
2023-07-04 08:22:16,152 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-04 08:22:16,153 [DEBUG] [Divider] Divider stopped serving
2023-07-04 08:22:16,154 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-04 08:22:16,155 [DEBUG] [Divider] Divider stopped serving
2023-07-04 08:22:16,155 [INFO] [Provisioner] provisioner stopped serving


In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.0831 - accuracy: 0.9753

Test accuracy: 97.5%
